In [ ]:
# predict total score in any given nhl game

In [ ]:
import pandas as pd


In [ ]:
# write final modeling data to excel
modeling_data = pd.read_excel(r'data/modeling_data.xlsx', header=0)

# inspect
modeling_data.info()
modeling_data.head()


In [ ]:
# set season as categorical
modeling_data['Season'] = modeling_data['Season'].astype('str')
modeling_data.info()

In [ ]:
modeling_data['Total_Score'].describe()

In [ ]:
# drop na to make life easier
modeling_data = modeling_data.dropna()

# trim data roughly to regular season games only
modeling_data = modeling_data[
    (modeling_data['Month'].isin(['October', 'November', 'December', 'January', 'February', 'March'])) # trim to regular season months with a smidge of oct pre-season
                              | 
    (modeling_data['Month'].isin(['April']) & modeling_data['Date'].dt.day <= 15) # include up to april 15th
]

# inspect
modeling_data.info()
modeling_data['Total_Score'].describe()

In [ ]:
# designate response variable
response_ = 'Total_Score'


In [ ]:
# find ideal sample size to test on all of 2025
samp_size_2025 = modeling_data[modeling_data['Season']=='2025'].shape[0] / modeling_data.shape[0]
print(f'Sample size for 2025 season: {samp_size_2025:.2%}')

modeling_data['Season'].value_counts()

In [ ]:
# create test and train data
test_start_date = '2025-11-07'

# list of drop cols that won't be used in modeling
drop_cols = ['Game_ID', 'Date']

# isolate train
train_data = modeling_data[modeling_data['Date'] < test_start_date]
X_train = train_data.drop(columns=drop_cols + [response_], axis=1)
y_train = train_data[response_]

# isolate test
test_data = modeling_data[modeling_data['Date'] >= test_start_date]
X_test = test_data.drop(columns=drop_cols + [response_], axis=1)
y_test = test_data[response_]

# inspect
X_train.info()
y_train.info()

In [ ]:
# import bambi
import bambi as bmb

# 1. Inspect Total_Score distribution
mu_runs = modeling_data['Total_Score'].mean()
sd_runs = modeling_data['Total_Score'].std()

print('mu_runs', mu_runs)
print('sd_runs', sd_runs)
print('var_runs', sd_runs ** 2)

# 3. Estimate overdispersion (only for Negative Binomial)
dispersion_alpha_est = (sd_runs**2 - mu_runs) / (mu_runs**2)
dispersion_alpha_est = max(dispersion_alpha_est, 1e-3)  # Avoid zero or negative
print('dispersion_alpha_est', dispersion_alpha_est)

# create priors dict
priors_ = {
    "Intercept": bmb.Prior("Normal", mu=log_mu, sigma=1.5),
    "DayOfWeek": bmb.Prior("Normal", mu=0, sigma=1),
    "Month": bmb.Prior("Normal", mu=0, sigma=1),
    "Year": bmb.Prior("Normal", mu=0, sigma=1),
    "time_of_day": bmb.Prior("Normal", mu=0, sigma=1),
    "Stadium_Indoor": bmb.Prior("Normal", mu=0, sigma=1),
    "Coors_Field": bmb.Prior("Normal", mu=0, sigma=1),

    # HalfNormal priors for standard deviations of group-level (random) effects
    "sd(Division_Interaction|Intercept)": bmb.Prior("HalfNormal", sigma=1),
    "sd(Home_Team|Intercept)": bmb.Prior("HalfNormal", sigma=1),
    "sd(Away_Team|Intercept)": bmb.Prior("HalfNormal", sigma=1),
    "sd(Team_Interaction|Intercept)": bmb.Prior("HalfNormal", sigma=1),
    # "sd(Win_Rate_Interaction|Intercept)": bmb.Prior("HalfNormal", sigma=1), # commenting out to see impact to model overall
    "sd(Home_Team:Year|Intercept)": bmb.Prior("HalfNormal", sigma=bmb.Prior("HalfNormal", sigma=1.0)),
    "sd(Away_Team:Year|Intercept)": bmb.Prior("HalfNormal", sigma=bmb.Prior("HalfNormal", sigma=1.0)),
    "sd(Team_Interaction:Year|Intercept)": bmb.Prior("HalfNormal", sigma=bmb.Prior("HalfNormal", sigma=1.0)),
    "sd(DayOfWeek:time_of_day|Intercept)": bmb.Prior("HalfNormal", sigma=bmb.Prior("HalfNormal", sigma=1.0)),

    # add priors for home vs away clusters
    "sd(Home_Cluster|Intercept)": bmb.Prior("HalfNormal", sigma=1),
    "sd(Away_Cluster|Intercept)": bmb.Prior("HalfNormal", sigma=1),
    "sd(Home_Cluster:Away_Cluster|Intercept)": bmb.Prior("HalfNormal", sigma=bmb.Prior("HalfNormal", sigma=1.0)),

    # Overdispersion prior for Negative Binomial
    "alpha": bmb.Prior("Exponential", lam=1 / dispersion_alpha_est)
}


In [ ]:
# import joblib

# # write model to pkl file
# joblib.dump(basic_model, 'model/model_total_score.pkl')
